In [1]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [2]:
with open('C:\\Users\\dylan\\OneDrive\\Documentos\\mensajeria\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl)

In [3]:
def generar_dim_hora() -> pd.DataFrame:
    
    segundos = pd.date_range(start='00:00:00', end='23:59:59', freq='1s')
    
    df = pd.DataFrame({'tiempo': segundos})
    
    # Clave sustituta: HHMMSS como entero (0 a 235959)
    df['key_hora'] = df['tiempo'].dt.strftime('%H%M%S').astype(int)
    
    # Atributos
    df['hora']         = df['tiempo'].dt.hour
    df['minutos']      = df['tiempo'].dt.minute
    df['segundos']     = df['tiempo'].dt.second
    df['hora_formato'] = df['tiempo'].dt.strftime('%H:%M:%S')
    
    # Franjas horarias
    df['franja'] = pd.cut(
        df['hora'],
        bins=  [0,  6, 12, 14, 18, 21, 24],
        labels=['madrugada', 'mañana', 'mediodía', 'tarde', 'noche', 'noche alta'],
        right=False
    )
    
    df['es_hora_habil'] = (df['hora'] >= 8) & (df['hora'] < 18)
    
    return df[['key_hora', 'hora', 'minutos', 'segundos',
               'hora_formato', 'franja', 'es_hora_habil']]


dim_hora = generar_dim_hora()
print(dim_hora.shape)   # (86400, 7) → 86.400 segundos en un día
print(dim_hora.tail())

(86400, 7)
       key_hora  hora  minutos  segundos hora_formato      franja  \
86395    235955    23       59        55     23:59:55  noche alta   
86396    235956    23       59        56     23:59:56  noche alta   
86397    235957    23       59        57     23:59:57  noche alta   
86398    235958    23       59        58     23:59:58  noche alta   
86399    235959    23       59        59     23:59:59  noche alta   

       es_hora_habil  
86395          False  
86396          False  
86397          False  
86398          False  
86399          False  


In [4]:
dim_hora.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86400 entries, 0 to 86399
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   key_hora       86400 non-null  int64   
 1   hora           86400 non-null  int32   
 2   minutos        86400 non-null  int32   
 3   segundos       86400 non-null  int32   
 4   hora_formato   86400 non-null  object  
 5   franja         86400 non-null  category
 6   es_hora_habil  86400 non-null  bool    
dtypes: bool(1), category(1), int32(3), int64(1), object(1)
memory usage: 2.5+ MB


In [5]:
dim_hora.to_sql('dim_hora', etl_conn, if_exists='replace', index=False)

400